In [3]:
pip install xgboost optuna shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 10.4 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

import optuna
import shap

In [6]:
train = pd.read_csv("TRAIN.csv")
test = pd.read_csv("TEST.csv")

print(train.shape)
print(test.shape)

train.head()

(26984, 48)
(10944, 48)


,F01,F02,F03,F04,F05,F06,F07,F08,F09,F10,...,F39,F40,F41,F42,F43,F44,F45,F46,F47,Class
0,0.185570,0.004568,0.005362,0.003335,0.005415,0.004895,0.012764,0.120138,0.140450,3.361753,...,0.041526,-0.230857,0.003310,0.042250,0.005975,0.002104,0.013878,0.001518,0.011347,0.0
1,0.369536,0.003983,0.003386,0.004902,0.007570,0.012136,0.118050,0.323925,0.132093,2.766117,...,-0.141285,-6.222857,0.834177,0.227968,0.018463,-0.020487,0.001246,0.007489,0.008724,1.0
2,0.602510,0.008442,0.012961,0.012870,0.046885,0.115401,0.065688,0.306677,0.498805,4.521201,...,0.011334,10.335251,-0.276614,-0.198900,-0.012756,0.014286,-0.001866,0.002687,0.013452,1.0
3,0.347957,0.064721,0.013611,0.011541,0.006492,0.008690,0.013192,0.164553,0.298665,3.170847,...,0.190479,2.864912,-1.921939,0.891690,1.108098,0.635431,0.081368,-0.000225,0.009166,0.0
4,0.233653,0.012217,0.010088,0.022095,0.026040,0.015062,0.016063,0.084648,0.213367,8.150943,...,0.203164,0.001812,-0.092731,0.005280,-0.213985,0.032195,0.002081,0.028930,-0.025912,1.0


In [32]:
X = train_clean.drop("Class", axis=1)

y = train_clean["Class"]

In [33]:
interaction_pairs = [
    ("F01","F02"),
    ("F03","F04"),
    ("F05","F06"),
    ("F07","F08"),
    ("F09","F10")
]

for f1, f2 in interaction_pairs:

    X[f"{f1}_{f2}_mul"] = X[f1] * X[f2]

    X[f"{f1}_{f2}_ratio"] = X[f1] / (X[f2] + 1e-6)

    test[f"{f1}_{f2}_mul"] = test[f1] * test[f2]

    test[f"{f1}_{f2}_ratio"] = test[f1] / (test[f2] + 1e-6)

In [34]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [35]:
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [36]:
importance = rf.feature_importances_

feature_importance = pd.Series(
    importance,
    index=X.columns
)

top_features = feature_importance.sort_values(
    ascending=False
).head(25)

selected_features = top_features.index

X_selected = X[selected_features]

In [37]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

xgb_base = XGBClassifier()

scores = cross_val_score(
    xgb_base,
    X_selected,
    y,
    cv=skf,
    scoring="f1",
    n_jobs=-1
)

print("CV Scores:", scores)
print("Mean F1:", scores.mean())

CV Scores: [0.97587512 0.97268017 0.97326709 0.96694411 0.97661233]
Mean F1: 0.9730757652321318


In [38]:
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators",200,800),
        "max_depth": trial.suggest_int("max_depth",3,10),
        "learning_rate": trial.suggest_float("learning_rate",0.01,0.2),
        "subsample": trial.suggest_float("subsample",0.6,1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree",0.6,1.0)
    }

    model = XGBClassifier(**params)

    score = cross_val_score(
        model,
        X_selected,
        y,
        cv=skf,
        scoring="f1",
        n_jobs=-1
    ).mean()

    return score


study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=50)

best_params = study.best_params

print("Best Parameters:", best_params)

[I 2026-03-05 10:35:20,719] A new study created in memory with name: no-name-9931506d-30be-462c-9217-e5cb02a8faa7
[I 2026-03-05 10:35:43,380] Trial 0 finished with value: 0.9798825248664244 and parameters: {'n_estimators': 436, 'max_depth': 9, 'learning_rate': 0.1333541131743334, 'subsample': 0.6950969793563336, 'colsample_bytree': 0.7646769293857804}. Best is trial 0 with value: 0.9798825248664244.
[I 2026-03-05 10:36:12,693] Trial 1 finished with value: 0.9657085242473006 and parameters: {'n_estimators': 335, 'max_depth': 10, 'learning_rate': 0.0163630679135683, 'subsample': 0.6982646305722173, 'colsample_bytree': 0.7391145718137875}. Best is trial 0 with value: 0.9798825248664244.
[I 2026-03-05 10:36:51,647] Trial 2 finished with value: 0.9717323622004427 and parameters: {'n_estimators': 474, 'max_depth': 10, 'learning_rate': 0.01582013295082982, 'subsample': 0.884792214686317, 'colsample_bytree': 0.6576465980608431}. Best is trial 0 with value: 0.9798825248664244.
[I 2026-03-05 10:

Best Parameters: {'n_estimators': 798, 'max_depth': 9, 'learning_rate': 0.10421133247184226, 'subsample': 0.9552210519732836, 'colsample_bytree': 0.8957048906124632}


In [39]:
xgb_best = XGBClassifier(**best_params)

xgb_best.fit(X_selected, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8957048906124632, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, feature_weights=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.10421133247184226,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=9, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=798, n_jobs=None,
              num_parallel_tree=None, ...)

In [40]:
test_features = test[selected_features]

predictions = xgb_best.predict(test_features)

In [41]:
submission = pd.DataFrame({
    "ID": test["ID"],
    "Class": predictions
})

submission.to_csv("FINAL.csv", index=False)

In [42]:
print(X_selected.shape)
print(y.shape)

(26983, 25)
(26983,)
